## Function Calling with Foundry Agents

### Installing Utilities and Libraries

In [ ]:
%pip install azure-ai-projects==2.7.0 openai==1.109.1 python-dotenv azure-identity jsonref

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("CHAT_MODEL_DEPLOYMENT_NAME")

### Setting up the Foundry Project Client

In [ ]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Creating the OpenAI Client

In [ ]:
openai_client = project_client.get_openai_client()

### Define the Function for Function Calling

In [ ]:
def get_current_weather(location: str) -> str:
    """
    Gets the current weather condition for a location.

    Args:
        location (string): The location, e.g. San Francisco, Mumbai etc.

    Returns:
        string: a string containing the weather information
    """
    
    import requests 
    
    response = requests.get(
        f"https://wttr.in/{location}",
        params={
            "format": "j1"
        }
    )

    response.raise_for_status()

    return str(response.json())

### Define the Function Calling Tool Capability

In [ ]:
from azure.ai.projects.models import Tool, FunctionTool, PromptAgentDefinition

func_tool = FunctionTool(
    name="get_current_weather",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The location to get the current weather for, e.g. San Francisco, Mumbai, London.",
            },
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    description="Get the current weather conditions for a specified location.",
    strict=True,
)

### Create your Agent

In [ ]:
agent_name = "function-calling-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are a helpful assistant that call user defined functions to answer user queries",
        tools=[func_tool],
    )
)

print(f"Created agent: {agent.name} with ID: {agent.id}")

### Create a Conversation Object for the Agent Chat System

In [ ]:
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

### Implement the Agent Loop

In [ ]:
user_query = "How is the weather in London currently?"

In [ ]:
from openai.types.responses.response_input_param import FunctionCallOutput, ResponseInputParam
import json

response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body = {
        "agent_reference": {
            "name": agent_name,
            "type": "agent_reference"
        }
    },
    input = user_query
)

input_list: ResponseInputParam = []
# Process function calls
for item in response.output:
    if item.type == "function_call":
        if item.name == "get_current_weather":
            # Execute the function logic for get_current_weather
            weather = get_current_weather(**json.loads(item.arguments))

            # Provide function call results to the model
            input_list.append(
                FunctionCallOutput(
                    type="function_call_output",
                    call_id=item.call_id,
                    output=json.dumps({"weather": weather}),
                )
            )

# Submit function results and get the final response
response = openai_client.responses.create(
    input=input_list,
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

print(f"Agent response: {response.output_text}")